## Install libraries

OpenAI API key is no longer needed as we are switching to open-source models. You can remove the `OPENAI_API_KEY` from your Colab secrets.

In [ ]:
!pip install -q -U youtube-transcript-api langchain-community langchain-huggingface langchain-core langchain-text-splitters faiss-cpu requests huggingface_hub


In [ ]:
from youtube_transcript_api import (
    YouTubeTranscriptApi,
    TranscriptsDisabled,
    NoTranscriptFound,
    VideoUnavailable,
)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings  # Updated import
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.prompts import PromptTemplate


In [ ]:
import os
from getpass import getpass

# Don't hardcode tokens in notebooks you might share or commit to git.
# If the env var isn't already set (e.g. via Colab secrets), prompt for it once.
if "HUGGINGFACEHUB_API_TOKEN" not in os.environ:
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = getpass(
        "Enter your Hugging Face access token (https://huggingface.co/settings/tokens): "
    )

# Base endpoint: talks to the model repo via HF's Inference Providers.
llm_endpoint = HuggingFaceEndpoint(
    repo_id="HuggingFaceH4/zephyr-7b-beta",
    task="conversational",   # most hosted chat models only support this task now
    temperature=0.5,
    max_new_tokens=512,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.03,
)

# Wrap it as a chat model. HF's Inference Providers route almost all chat-tuned
# models (zephyr, Mistral-Instruct, Llama-Instruct, etc.) through the
# "conversational" / chat-completions API rather than raw text-generation.
# Calling llm_endpoint.invoke(...) directly raises:
#   ValueError: Model ... is not supported for task text-generation ...
# ChatHuggingFace fixes this by calling the correct chat-completions endpoint.
llm = ChatHuggingFace(llm=llm_endpoint)


## Step 1a - Indexing (Document Ingestion)

In [ ]:
video_id = "Gfr50f6ZBvo"  # only the ID, not full URL

transcript = None

try:
    # 1. Initialize the API client
    yt_api = YouTubeTranscriptApi()

    # 2. Fetch the transcript and convert it to raw data (list of dictionaries)
    transcript_list = yt_api.fetch(video_id, languages=["en"]).to_raw_data()

    # 3. Flatten it to plain text
    transcript = " ".join(chunk["text"] for chunk in transcript_list)

    if not transcript.strip():
        raise ValueError("Transcript was fetched but is empty.")

    print(transcript[:500] + ("..." if len(transcript) > 500 else ""))

except TranscriptsDisabled:
    print(f"Captions are disabled for video '{video_id}'.")
except NoTranscriptFound:
    print(f"No English transcript found for video '{video_id}'. "
          f"Try a different `languages` list (e.g. languages=['en', 'en-US']).")
except VideoUnavailable:
    print(f"Video '{video_id}' is unavailable (private, deleted, or invalid ID).")
except Exception as e:
    print(f"Unexpected error while fetching transcript: {e}")

if transcript is None:
    raise RuntimeError(
        "transcript was not loaded -- fix the error above before continuing. "
        "All later cells depend on this variable."
    )


In [ ]:
transcript_list

[{'text': 'the following is a conversation with',
  'start': 0.08,
  'duration': 3.44},
 {'text': 'demus hasabis', 'start': 1.76, 'duration': 4.96},
 {'text': 'ceo and co-founder of deepmind', 'start': 3.52, 'duration': 5.119},
 {'text': 'a company that has published and builds',
  'start': 6.72,
  'duration': 4.48},
 {'text': 'some of the most incredible artificial',
  'start': 8.639,
  'duration': 4.561},
 {'text': 'intelligence systems in the history of',
  'start': 11.2,
  'duration': 4.8},
 {'text': 'computing including alfred zero that',
  'start': 13.2,
  'duration': 3.68},
 {'text': 'learned', 'start': 16.0, 'duration': 2.96},
 {'text': 'all by itself to play the game of gold',
  'start': 16.88,
  'duration': 4.559},
 {'text': 'better than any human in the world and',
  'start': 18.96,
  'duration': 5.6},
 {'text': 'alpha fold two that solved protein',
  'start': 21.439,
  'duration': 4.241},
 {'text': 'folding', 'start': 24.56, 'duration': 4.16},
 {'text': 'both tasks consider

## Step 1b - Indexing (Text Splitting)

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

if len(chunks) == 0:
    raise ValueError("No chunks were created -- check that `transcript` is non-empty.")

print(f"Created {len(chunks)} chunks.")


In [ ]:
len(chunks)

168

In [ ]:
# Peek at a chunk near the middle of the document (safe for any chunk count)
sample_idx = min(100, len(chunks) - 1)
chunks[sample_idx]


## Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [ ]:
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(chunks, embeddings)

/tmp/ipykernel_62916/4273299600.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
vector_store.index_to_docstore_id

{0: '5e2256c8-ecc7-403c-acf8-45b74c02bd30',
 1: '1ec372f6-7a7b-4270-b254-ecf98687ed64',
 2: '156525c5-8eb6-49a7-9d26-85dd6c85103e',
 3: '56c658ce-9fb7-4e05-8228-a6f1a4444eb9',
 4: '340082d2-aa89-4aec-95a4-1ecdb5c571cf',
 5: 'e80bb2a2-a048-465e-9dbf-dbb33cd05c79',
 6: 'f43af49b-7c6c-4675-93ff-aa637005c8ae',
 7: '49c47119-88f8-4bca-aecd-53d8fb043a58',
 8: 'eb2b4c08-f9ef-4acb-a159-067c302015a9',
 9: '67efadec-c13f-4a6a-a48c-99f72d8d6d42',
 10: 'efad25ae-4b8d-4a56-833b-19f86625951a',
 11: 'e4682fa5-5f75-46aa-a5ae-364390294142',
 12: 'a59ddd00-fb51-4476-940c-307555db8c34',
 13: 'ecfc26a3-e241-43c8-ba22-22d3cff9b2ea',
 14: '9da56c97-8d0d-4196-a8a4-012613fb8c9c',
 15: '1776edab-b3aa-41b6-8d85-2bebfbeb87e4',
 16: 'a9c92da1-868e-40d5-8c44-f97f4269f2cb',
 17: '3eb274dc-3514-4519-8ad2-e1d755a6f2ca',
 18: 'b82c9173-f381-4bd0-8ada-4fa893978680',
 19: 'd8f0bd5a-381f-48da-9782-78873135f644',
 20: 'af8fa436-3fe5-4606-a0ea-1ba099ca1de9',
 21: '50c4392e-e4a2-4eef-b38f-fa222ecda57b',
 22: '8eafd018-0307-

In [ ]:
# Use a real ID from the index we just built (don't hardcode a UUID --
# it changes every time the vector store is regenerated, and a stale one
# would silently return [] instead of erroring).
sample_id = vector_store.index_to_docstore_id[0]
vector_store.get_by_ids([sample_id])


## Step 2 - Retrieval

In [ ]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [ ]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7e2c18be8bf0>, search_kwargs={'k': 4})

In [ ]:
retriever.invoke('What is deepmind')

[Document(id='75ada205-fa3d-4f99-b62b-809e6c779c65', metadata={}, page_content="and how it works this is tough to uh ask you this question because you probably will say it's everything but let's let's try let's try to think to this because you're in a very interesting position where deepmind is the place of some of the most uh brilliant ideas in the history of ai but it's also a place of brilliant engineering so how much of solving intelligence this big goal for deepmind how much of it is science how much is engineering so how much is the algorithms how much is the data how much is the hardware compute infrastructure how much is it the software computer infrastructure yeah um what else is there how much is the human infrastructure and like just the humans interact in certain kinds of ways in all the space of all those ideas how much does maybe like philosophy how much what's the key if um uh if if you were to sort of look back like if we go forward 200 years look back what was the key 

## Step 3 - Augmentation

In [ ]:
from textwrap import dedent

prompt = PromptTemplate(
    template=dedent("""
        You are a helpful assistant.
        Answer ONLY from the provided transcript context.
        If the context is insufficient, just say you don't know.

        {context}
        Question: {question}
    """).strip(),
    input_variables=['context', 'question']
)


In [ ]:
question          = "is the topic of nuclear fusion discussed in this video? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question)

In [ ]:
retrieved_docs

[Document(id='26520b2d-dc45-43c3-8f53-6f69db188dda', metadata={}, page_content="in this case in fusion we we collaborated with epfl in switzerland the swiss technical institute who are amazing they have a test reactor that they were willing to let us use which you know i double checked with the team we were going to use carefully and safely i was impressed they managed to persuade them to let us use it and um and it's a it's an amazing test reactor they have there and they try all sorts of pretty crazy experiments on it and um the the the what we tend to look at is if we go into a new domain like fusion what are all the bottleneck problems uh like thinking from first principles you know what are all the bottleneck problems that are still stopping fusion working today and then we look at we you know we get a fusion expert to tell us and then we look at those bottlenecks and we look at the ones which ones are amenable to our ai methods today yes right and and and then and would be intere

In [ ]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"in this case in fusion we we collaborated with epfl in switzerland the swiss technical institute who are amazing they have a test reactor that they were willing to let us use which you know i double checked with the team we were going to use carefully and safely i was impressed they managed to persuade them to let us use it and um and it's a it's an amazing test reactor they have there and they try all sorts of pretty crazy experiments on it and um the the the what we tend to look at is if we go into a new domain like fusion what are all the bottleneck problems uh like thinking from first principles you know what are all the bottleneck problems that are still stopping fusion working today and then we look at we you know we get a fusion expert to tell us and then we look at those bottlenecks and we look at the ones which ones are amenable to our ai methods today yes right and and and then and would be interesting from a research perspective from our point of view from an ai point of\n\

In [ ]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [ ]:
final_prompt

StringPromptValue(text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      in this case in fusion we we collaborated with epfl in switzerland the swiss technical institute who are amazing they have a test reactor that they were willing to let us use which you know i double checked with the team we were going to use carefully and safely i was impressed they managed to persuade them to let us use it and um and it's a it's an amazing test reactor they have there and they try all sorts of pretty crazy experiments on it and um the the the what we tend to look at is if we go into a new domain like fusion what are all the bottleneck problems uh like thinking from first principles you know what are all the bottleneck problems that are still stopping fusion working today and then we look at we you know we get a fusion expert to tell us and then we look at those bottlenecks and we 

## Step 4 - Generation

In [ ]:
# `llm` is a ChatHuggingFace model, so invoking it returns an AIMessage object.
# Print `.content` to get the clean text instead of the full message repr.
answer = llm.invoke(final_prompt)
print(answer.content)


## Building a Chain

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [ ]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [ ]:
parallel_chain.invoke('who is Demis')

In [ ]:
parser = StrOutputParser()

In [ ]:
main_chain = parallel_chain | prompt | llm | parser

In [ ]:
main_chain.invoke('Can you summarize the video')